# 04 · Model Training

**Goal:** train every model on every backtest fold and save the predictions and the trained XGBoost model.

| Model | Idea |
| --- | --- |
| Persistence | PM2.5 in 6 hours = PM2.5 now |
| Seasonal naive | PM2.5 in 6 hours = PM2.5 at the same hour yesterday |
| XGBoost | 300 gradient-boosted trees on 31 features |

In [1]:
import os
import sys

# Find the project folder (the one that contains config.py) and work from there
ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, "config.py")):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
sys.path.append(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import config

FIG = "reports/figures"
OUT = "data/GeneratedOutput/03.ModelTraining"
os.makedirs(FIG, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

In [2]:
import time
from data_loader import load_raw_data, clean_data
from features import build_features, get_feature_columns
from backtest import make_folds, compute_metrics, observed_only
from baselines import persistence_forecast, seasonal_naive_forecast
from xgboost_model import train_and_predict_xgboost

features = build_features(clean_data(load_raw_data(config.RAW_FILE)))
folds = make_folds(features)
print("Folds:", len(folds), "| Features:", len(get_feature_columns(features)))

Folds: 4 | Features: 31


## Train on each fold

In [3]:
all_predictions = []
log_rows = []

for fold, train, test in folds:
    start = time.time()
    xgb_preds, model = train_and_predict_xgboost(train, test)
    seconds = round(time.time() - start, 1)

    preds = pd.DataFrame(index=test.index)
    preds["forecast_for"] = test.index + pd.Timedelta(hours=config.HORIZON)
    preds["fold"] = fold
    preds["actual"] = test["target"]
    preds["target_observed"] = test["target_observed"]
    preds["Persistence"] = persistence_forecast(test)
    preds["Seasonal naive"] = seasonal_naive_forecast(test)
    preds["XGBoost"] = xgb_preds
    all_predictions.append(preds)

    scored = observed_only(preds)
    xgb_scores = compute_metrics(scored["actual"], scored["XGBoost"])
    log_rows.append({"fold": fold, "train_rows": len(observed_only(train)), "test_rows": len(scored),
                     "training_seconds": seconds, **{f"xgb_{k}": v for k, v in xgb_scores.items()}})
    print(f"Fold {fold}: trained in {seconds}s | XGBoost {xgb_scores}")

    # Keep the model from the last fold (trained on the most data) for feature importance
    last_model = model

Fold 1: trained in 0.6s | XGBoost {'MAE': 42.96, 'RMSE': 70.61, 'R2': 0.598}


Fold 2: trained in 0.6s | XGBoost {'MAE': 32.24, 'RMSE': 48.26, 'R2': 0.635}


Fold 3: trained in 0.7s | XGBoost {'MAE': 36.11, 'RMSE': 56.36, 'R2': 0.667}


Fold 4: trained in 0.7s | XGBoost {'MAE': 31.72, 'RMSE': 47.96, 'R2': 0.708}


## Save the outputs

In [4]:
predictions = pd.concat(all_predictions)
predictions.index.name = "forecast_made_at"
predictions.to_csv(f"{OUT}/predictions.csv")

training_log = pd.DataFrame(log_rows)
training_log.to_csv(f"{OUT}/training_log.csv", index=False)

last_model.save_model(f"{OUT}/xgboost_fold4.json")

print("Saved", len(predictions), "predictions, the training log and the fold-4 XGBoost model")
training_log

Saved 17514 predictions, the training log and the fold-4 XGBoost model


,fold,train_rows,test_rows,training_seconds,xgb_MAE,xgb_RMSE,xgb_R2
0,1,24388,4309,0.6,42.96,70.61,0.598
1,2,28697,4369,0.6,32.24,48.26,0.635
2,3,33066,4311,0.7,36.11,56.36,0.667
3,4,37377,4344,0.7,31.72,47.96,0.708


Training is fast (a few seconds per fold), so the whole backtest can be re-run any time. The evaluation notebooks read these saved files instead of training again.